## 1. Imports and the single predefined input

The fixed candidate panel comes from `agnostik.candidates`. `ARTICLE_QUERIES` pairs the COAD cancer expression with each predefined gene using `AND`; `ARTICLE_QUERY` combines those clauses for one search and one COAD output folder. `MAX_N` controls both the PubMed result request and the maximum number of complete articles saved.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

from IPython.display import FileLink, HTML, Markdown, display
from agnostik.candidates import PRESELECTED_CANDIDATES

TCGA_PROJECT = "TCGA-COAD"
CANCER_QUERY = "colon adenocarcinoma"
CANCER_ARTICLE_QUERY = "(COAD[Title/Abstract] OR \"colon adenocarcinoma\"[Title/Abstract] OR \"colorectal\"[Title/Abstract])"
GENES = PRESELECTED_CANDIDATES
ARTICLE_QUERIES = {
    gene: f"{CANCER_ARTICLE_QUERY} AND {gene}[Title/Abstract]"
    for gene in GENES
}
ARTICLE_QUERY = " OR ".join(f"({query})" for query in ARTICLE_QUERIES.values())
TRIAL_QUERY = CANCER_QUERY
MAX_N = 300

print(f"TCGA project: {TCGA_PROJECT}")
print(f"Shared skill input: {CANCER_QUERY!r}")
print(f"Genes: {', '.join(GENES)}")
for gene, query in ARTICLE_QUERIES.items():
    print(f"{gene}: {query}")
print(f"Combined article query requests at most {MAX_N} articles.")
print(f"Clinical-trial query: {TRIAL_QUERY!r}")

TCGA project: TCGA-COAD
Shared skill input: 'colon adenocarcinoma'
Genes: EGFR, ERBB2, KRAS, MYC, WRN, PRMT5
EGFR: (COAD[Title/Abstract] OR "colon adenocarcinoma"[Title/Abstract] OR "colorectal"[Title/Abstract]) AND EGFR[Title/Abstract]
ERBB2: (COAD[Title/Abstract] OR "colon adenocarcinoma"[Title/Abstract] OR "colorectal"[Title/Abstract]) AND ERBB2[Title/Abstract]
KRAS: (COAD[Title/Abstract] OR "colon adenocarcinoma"[Title/Abstract] OR "colorectal"[Title/Abstract]) AND KRAS[Title/Abstract]
MYC: (COAD[Title/Abstract] OR "colon adenocarcinoma"[Title/Abstract] OR "colorectal"[Title/Abstract]) AND MYC[Title/Abstract]
WRN: (COAD[Title/Abstract] OR "colon adenocarcinoma"[Title/Abstract] OR "colorectal"[Title/Abstract]) AND WRN[Title/Abstract]
PRMT5: (COAD[Title/Abstract] OR "colon adenocarcinoma"[Title/Abstract] OR "colorectal"[Title/Abstract]) AND PRMT5[Title/Abstract]
Combined article query requests at most 300 articles.
Clinical-trial query: 'colon adenocarcinoma'


## 2. Locate and verify the installed ClawBio skills

The repository pins `clawbio==0.6.1`. This cell resolves the package instead of assuming a platform-specific environment path.

In [2]:
import clawbio

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not find the project root (pyproject.toml).")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
CLAWBIO_ROOT = Path(clawbio.__file__).resolve().parent
SKILLS_ROOT = CLAWBIO_ROOT / "skills"

PUBMED_SKILL = SKILLS_ROOT / "pubmed-summariser"
TRIAL_SKILL = SKILLS_ROOT / "clinical-trial-finder"
PUBMED_SCRIPT = PUBMED_SKILL / "pubmed_summariser.py"
TRIAL_SCRIPT = TRIAL_SKILL / "clinical_trial_finder.py"

for skill_name, skill_dir, script in [
    ("pubmed-summariser", PUBMED_SKILL, PUBMED_SCRIPT),
    ("clinical-trial-finder", TRIAL_SKILL, TRIAL_SCRIPT),
]:
    assert (skill_dir / "SKILL.md").is_file(), f"Missing metadata for {skill_name}"
    assert script.is_file(), f"Missing executable script for {skill_name}"
    frontmatter = (skill_dir / "SKILL.md").read_text(encoding="utf-8").split("---", 2)[1]
    assert f"name: {skill_name}" in frontmatter, f"Unexpected skill name in {skill_dir / 'SKILL.md'}"
    print(f"verified {skill_name}: {script}")

verified pubmed-summariser: C:\Users\neu\PycharmProjects\agnostik\.venv\Lib\site-packages\clawbio\skills\pubmed-summariser\pubmed_summariser.py
verified clinical-trial-finder: C:\Users\neu\PycharmProjects\agnostik\.venv\Lib\site-packages\clawbio\skills\clinical-trial-finder\clinical_trial_finder.py


## 3. Small runner for the ClawBio command-line skills

In [3]:
RESULTS_ROOT = PROJECT_ROOT / "results" / "clawbio_skill_trial" / TCGA_PROJECT.lower()
PUBMED_OUTPUT = RESULTS_ROOT / "pubmed"
FULL_TEXT_OUTPUT = RESULTS_ROOT / "full_text_articles"
FULL_TEXT_ARTIFACTS = RESULTS_ROOT / "full_text_article_artifacts"
TRIAL_OUTPUT = RESULTS_ROOT / "clinical_trials"

def run_skill(script: Path, *arguments: str) -> subprocess.CompletedProcess[str]:
    command = [sys.executable, str(script), *map(str, arguments)]
    print("Running:", subprocess.list2cmdline(command))
    completed = subprocess.run(
        command,
        cwd=PROJECT_ROOT,
        text=True,
        capture_output=True,
        timeout=180,
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr, file=sys.stderr)
    completed.check_returncode()
    return completed

## 4. Download up to `MAX_N` complete open-access articles

This is intentionally separate from ClawBio's excerpt report. It searches PubMed Central for the gene–cancer-treatment query and requires the Open Access filter. `full_text_articles/` is a flat Parseltongue source directory containing only one clean `.txt` file per complete article. Human-readable HTML, authoritative JATS XML, the manifest, and the index go into the sibling `full_text_article_artifacts/` directory. It never substitutes an abstract-only record when full text is unavailable.

In [4]:
from agnostik.pmc_full_text import download_open_access_articles

# Set this to a real contact address if you run many NCBI requests.
NCBI_EMAIL = "agnostik@example.com"

full_text_articles = download_open_access_articles(
    query=ARTICLE_QUERY,
    max_articles=MAX_N,
    output_dir=FULL_TEXT_OUTPUT,
    artifacts_dir=FULL_TEXT_ARTIFACTS,
    email=NCBI_EMAIL,
)
print(f"Saved {len(full_text_articles)} complete article(s); requested at most {MAX_N}.")
print(f"Parseltongue source directory: {FULL_TEXT_OUTPUT}")
if len(full_text_articles) < MAX_N:
    print("Fewer were saved because PMC Open Access had fewer qualifying full texts.")

Saved 300 complete article(s); requested at most 300.
Parseltongue source directory: C:\Users\neu\PycharmProjects\agnostik\results\clawbio_skill_trial\tcga-coad\full_text_articles


In [5]:
rows = ["| Article | IDs | Saved files |", "|---|---|---|"]
for article in full_text_articles[:10]:
    ids = ", ".join(value for value in (article.pmc_id, f"PMID {article.pmid}" if article.pmid else "", article.doi) if value)
    safe_title = article.title.replace("|", "\\|")
    rows.append(f"| {safe_title} | {ids} | Parseltongue TXT + HTML + source XML |")
display(Markdown("\n".join(rows)))
if full_text_articles:
    display(FileLink(FULL_TEXT_ARTIFACTS / "index.html"))
    for article in full_text_articles:
        display(FileLink(article.source_path))

| Article | IDs | Saved files |
|---|---|---|
| Changes in Colorectal Carcinoma Genomes under Anti-EGFR Therapy Identified by Whole-Genome Plasma DNA Sequencing | PMC3967949, PMID 24676216, 10.1371/journal.pgen.1004271 | Parseltongue TXT + HTML + source XML |
| Comprehensive genomic sequencing detects important genetic differences between right-sided and left-sided colorectal cancer | PMC5706819, PMID 29212173, 10.18632/oncotarget.20510 | Parseltongue TXT + HTML + source XML |
| PRMT5 Inhibition as a Potential Strategy for KRAS Mutant CRC: Downstream Mediators of the PRMT5–KRAS Crosstalk | PMC12384206, PMID 40864819, 10.3390/cimb47080665 | Parseltongue TXT + HTML + source XML |
| Higher frequency of subclonal anti-EGFR resistance mutations in post-treatment samples from patients with colorectal cancer liver metastases following anti-EGFR-based conversion chemotherapy | PMC13336157, PMID 42406708, 10.1371/journal.pone.0351243 | Parseltongue TXT + HTML + source XML |
| Establishment, biobanking, and multi-omics characterization of 41 patient-derived colorectal cancer organoids: MYC/PRC classification | PMC13316694, PMID 42349318, 10.1016/j.tranon.2026.102878 | Parseltongue TXT + HTML + source XML |
| ERBB2 and KRAS alterations mediate response to EGFR inhibitors in early stage gallbladder cancer | PMC6378102, PMID 30304546, 10.1002/ijc.31916 | Parseltongue TXT + HTML + source XML |
| Molecular characterization of ERBB2-amplified colorectal cancer identifies potential mechanisms of resistance to targeted therapies: a report of two instructive cases | PMC5880263, PMID 29438965, 10.1101/mcs.a002535 | Parseltongue TXT + HTML + source XML |
| Comparison of the Data of a Next-Generation Sequencing Panel from K-MASTER Project with That of Orthogonal Methods for Detecting Targetable Genetic Alterations | PMC8756135, PMID 34015890, 10.4143/crt.2021.218 | Parseltongue TXT + HTML + source XML |
| Gene mutations in stool from gastric and colorectal neoplasia patients by next-generation sequencing | PMC5743500, PMID 29307989, 10.3748/wjg.v23.i47.8291 | Parseltongue TXT + HTML + source XML |
| Clinical application of targeted next-generation sequencing for colorectal cancer patients: a multicentric Belgian experience | PMC5945518, PMID 29755687, 10.18632/oncotarget.25099 | Parseltongue TXT + HTML + source XML |

C:\Users\neu\PycharmProjects\agnostik\results\clawbio_skill_trial\tcga-coad\full_text_article_artifacts\index.html

C:\Users\neu\PycharmProjects\agnostik\results\clawbio_skill_trial\tcga-coad\full_text_articles\PMC3967949.txt

C:\Users\neu\PycharmProjects\agnostik\results\clawbio_skill_trial\tcga-coad\full_text_articles\PMC5706819.txt

C:\Users\neu\PycharmProjects\agnostik\results\clawbio_skill_trial\tcga-coad\full_text_articles\PMC12384206.txt

...

C:\Users\neu\PycharmProjects\agnostik\results\clawbio_skill_trial\tcga-coad\full_text_articles\PMC8987589.txt

C:\Users\neu\PycharmProjects\agnostik\results\clawbio_skill_trial\tcga-coad\full_text_articles\PMC5328292.txt

C:\Users\neu\PycharmProjects\agnostik\results\clawbio_skill_trial\tcga-coad\full_text_articles\PMC3596735.txt

C:\Users\neu\PycharmProjects\agnostik\results\clawbio_skill_trial\tcga-coad\full_text_articles\PMC11246022.txt

C:\Users\neu\PycharmProjects\agnostik\results\clawbio_skill_trial\tcga-coad\full_text_articles\PMC11986171.txt

## 5. Run `pubmed-summariser`

This uses the skill's `--query`, `--max-results`, and `--output` interface.

In [ ]:
run_skill(
    PUBMED_SCRIPT,
    "--query", ARTICLE_QUERY,
    "--max-results", str(MAX_N),
    "--output", str(PUBMED_OUTPUT),
)

In [ ]:
pubmed_report = PUBMED_OUTPUT / "report.html"
assert pubmed_report.is_file(), "The PubMed skill did not create report.html"
display(HTML(pubmed_report.read_text(encoding="utf-8")))
display(FileLink(pubmed_report))

## 6. Run `clinical-trial-finder`

This uses condition-query mode (`--query`). Trial status is deliberately not filtered: terminated and withdrawn studies are evidence too.

In [6]:
run_skill(
    TRIAL_SCRIPT,
    "--query", TRIAL_QUERY,
    "--max-results", "10",
    "--output", str(TRIAL_OUTPUT),
)

Running: C:\Users\neu\PycharmProjects\agnostik\.venv\Scripts\python.exe C:\Users\neu\PycharmProjects\agnostik\.venv\Lib\site-packages\clawbio\skills\clinical-trial-finder\clinical_trial_finder.py --query "colon adenocarcinoma" --max-results 10 --output C:\Users\neu\PycharmProjects\agnostik\results\clawbio_skill_trial\tcga-coad\clinical_trials
Querying ClinicalTrials.gov: 'colon adenocarcinoma'
Found 10 trials (1 recruiting)
Report  -> C:\Users\neu\PycharmProjects\agnostik\results\clawbio_skill_trial\tcga-coad\clinical_trials\report.md
HTML    -> C:\Users\neu\PycharmProjects\agnostik\results\clawbio_skill_trial\tcga-coad\clinical_trials\report.html
Summary -> C:\Users\neu\PycharmProjects\agnostik\results\clawbio_skill_trial\tcga-coad\clinical_trials\summary.json
CSV     -> C:\Users\neu\PycharmProjects\agnostik\results\clawbio_skill_trial\tcga-coad\clinical_trials\tables\trials.csv
Chart   -> C:\Users\neu\PycharmProjects\agnostik\results\clawbio_skill_trial\tcga-coad\clinical_trials\figu

CompletedProcess(args=['C:\\Users\\neu\\PycharmProjects\\agnostik\\.venv\\Scripts\\python.exe', 'C:\\Users\\neu\\PycharmProjects\\agnostik\\.venv\\Lib\\site-packages\\clawbio\\skills\\clinical-trial-finder\\clinical_trial_finder.py', '--query', 'colon adenocarcinoma', '--max-results', '10', '--output', 'C:\\Users\\neu\\PycharmProjects\\agnostik\\results\\clawbio_skill_trial\\tcga-coad\\clinical_trials'], returncode=0, stdout="Querying ClinicalTrials.gov: 'colon adenocarcinoma'\nFound 10 trials (1 recruiting)\nReport  -> C:\\Users\\neu\\PycharmProjects\\agnostik\\results\\clawbio_skill_trial\\tcga-coad\\clinical_trials\\report.md\nHTML    -> C:\\Users\\neu\\PycharmProjects\\agnostik\\results\\clawbio_skill_trial\\tcga-coad\\clinical_trials\\report.html\nSummary -> C:\\Users\\neu\\PycharmProjects\\agnostik\\results\\clawbio_skill_trial\\tcga-coad\\clinical_trials\\summary.json\nCSV     -> C:\\Users\\neu\\PycharmProjects\\agnostik\\results\\clawbio_skill_trial\\tcga-coad\\clinical_trials\

In [7]:
trial_summary_path = TRIAL_OUTPUT / "summary.json"
trial_report_path = TRIAL_OUTPUT / "report.md"
assert trial_summary_path.is_file(), "The trial skill did not create summary.json"
assert trial_report_path.is_file(), "The trial skill did not create report.md"

trial_data = json.loads(trial_summary_path.read_text(encoding="utf-8"))
display(Markdown(
    f"**Query:** `{trial_data['query']}`  \
"
    f"**Trials returned:** {trial_data['total']}  \
"
    f"**Recruiting:** {trial_data['recruiting']}"
))

rows = ["| NCT ID | Status | Phase | Title |", "|---|---|---|---|"]
for trial in trial_data["trials"]:
    nct_id = trial["nct_id"]
    title = trial["title"].replace("|", "\\|")
    rows.append(
        f"| [{nct_id}](https://clinicaltrials.gov/study/{nct_id}) "
        f"| {trial['status']} | {trial['phase'] or 'N/A'} | {title} |"
    )
display(Markdown("\n".join(rows)))
display(FileLink(trial_report_path))

**Query:** `colon adenocarcinoma`  **Trials returned:** 10  **Recruiting:** 1

| NCT ID | Status | Phase | Title |
|---|---|---|---|
| [NCT06363123](https://clinicaltrials.gov/study/NCT06363123) | UNKNOWN | N/A | Plasma Metabolic Biomarkers for Multi-Cancer Diagnosis |
| [NCT03724851](https://clinicaltrials.gov/study/NCT03724851) | COMPLETED | PHASE1 / PHASE2 | Vactosertib in Combination with Pembrolizumab in Metastatic Colorectal or Gastric Cancer |
| [NCT03871959](https://clinicaltrials.gov/study/NCT03871959) | COMPLETED | PHASE1 | Pembrolizumab In Combination With Debio 1143 In Pancreatic and Colorectal Advanced/Metastatic Adenocarcinoma |
| [NCT03373188](https://clinicaltrials.gov/study/NCT03373188) | COMPLETED | PHASE1 | VX15/2503 and Immunotherapy in Resectable Pancreatic and Colorectal Cancer |
| [NCT04510129](https://clinicaltrials.gov/study/NCT04510129) | RECRUITING | N/A | A Multicenter Cancer Biospecimen Collection Study |
| [NCT05462717](https://clinicaltrials.gov/study/NCT05462717) | ACTIVE_NOT_RECRUITING | PHASE1 | Dose Escalation and Dose Expansion Study of RMC-6291 Monotherapy in Subjects With Advanced KRASG12C Mutant Solid Tumors |
| [NCT03929666](https://clinicaltrials.gov/study/NCT03929666) | COMPLETED | PHASE2 | A Safety and Efficacy Study of ZW25 (Zanidatamab) Plus Combination Chemotherapy in HER2-expressing Gastrointestinal Cancers, Including Gastroesophageal Adenocarcinoma, Biliary Tract Cancer, and Colorectal Cancer |
| [NCT06524362](https://clinicaltrials.gov/study/NCT06524362) | UNKNOWN | NA | Effect of Pelvic Rehabilitation After Low Anterior Resection for Cancer Rectum. - A Randomised Controlled Trial |
| [NCT01417494](https://clinicaltrials.gov/study/NCT01417494) | COMPLETED | PHASE2 | 1st Line Chemotherapy Alone or With Bevacizumab in Treating Older Patients With Metastatic Colorectal Cancer |
| [NCT00007826](https://clinicaltrials.gov/study/NCT00007826) | UNKNOWN | PHASE1 / PHASE2 | Monoclonal Antibody Therapy and/or Vaccine Therapy in Treating Patients With Locally Advanced or Metastatic Colorectal Cancer |

C:\Users\neu\PycharmProjects\agnostik\results\clawbio_skill_trial\tcga-coad\clinical_trials\report.md

## 7. Generated artifacts

In [8]:
for path in sorted(RESULTS_ROOT.rglob("*")):
    if path.is_file():
        print(path.relative_to(PROJECT_ROOT))

results\clawbio_skill_trial\tcga-coad\clinical_trials\checksums.sha256
results\clawbio_skill_trial\tcga-coad\clinical_trials\commands.sh
results\clawbio_skill_trial\tcga-coad\clinical_trials\figures\phase_distribution.png
results\clawbio_skill_trial\tcga-coad\clinical_trials\report.html
results\clawbio_skill_trial\tcga-coad\clinical_trials\report.md
results\clawbio_skill_trial\tcga-coad\clinical_trials\summary.json
results\clawbio_skill_trial\tcga-coad\clinical_trials\tables\trials.csv
results\clawbio_skill_trial\tcga-coad\full_text_article_artifacts\html\PMC10037609.html
results\clawbio_skill_trial\tcga-coad\full_text_article_artifacts\html\PMC10045351.html
results\clawbio_skill_trial\tcga-coad\full_text_article_artifacts\html\PMC10181928.html
results\clawbio_skill_trial\tcga-coad\full_text_article_artifacts\html\PMC10190927.html
results\clawbio_skill_trial\tcga-coad\full_text_article_artifacts\html\PMC10196928.html
results\clawbio_skill_trial\tcga-coad\full_text_article_artifacts\htm

---

**Research-use note:** ClawBio is a research and educational tool, not a medical device. Trial status alone does not establish efficacy or safety, and trial eligibility must be checked in the registry with a qualified clinician.